# Fine-tune PhoBERT cho bài toán Phân loại Khiếu nại

Notebook này fine-tune mô hình PhoBERT (`vinai/phobert-base-v2`) cho bài toán phân loại văn bản nhị phân: 0 = Bình thường, 1 = Khiếu nại.

**Dữ liệu:** `data/processed/shopee_mapped.csv`  
**Tiền xử lý:** `clean_vietnamese_text` từ `src.utils.utils`  
**Huấn luyện:** Hugging Face `Trainer`

## Bước 1: Cài đặt và Import thư viện

In [ ]:
# Cell 1: Cài đặt thư viện cần thiết cho Colab
# Nếu chạy trên Colab, hãy bật cell này trước khi huấn luyện
!pip -q install transformers datasets evaluate accelerate

## Bước 2: Tải, làm sạch và chia dữ liệu

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

# Thêm đường dẫn gốc để import hàm clean_vietnamese_text
ROOT_DIR = os.path.abspath('..')
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.utils.utils import clean_vietnamese_text

# Thiết lập seed để kết quả ổn định
set_seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Cấu hình device (Trainer sẽ tự dùng GPU nếu có)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[OK] Device hiện tại: {device}')
print(f'[INFO] Root directory: {ROOT_DIR}')

In [ ]:
# Đọc dữ liệu gốc
csv_path = os.path.join('..', 'data', 'processed', 'shopee_mapped.csv')
df = pd.read_csv(csv_path, encoding='utf-8-sig')
print(f'[INFO] Số dòng ban đầu: {len(df)}')

# Xóa các dòng NaN ở cột review
before_na = df['review'].isna().sum()
df = df.dropna(subset=['review']).reset_index(drop=True)
print(f'[INFO] Số NaN ở cột review trước khi xóa: {before_na}')
print(f'[OK] Số dòng còn lại sau khi xóa NaN: {len(df)}')

# Làm sạch văn bản
df['cleaned_text'] = df['review'].astype(str).apply(clean_vietnamese_text)
print('[OK] Đã tạo cột cleaned_text')

# Chia train/test theo stratify
train_df, eval_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['complaint_label']
)

print(f'[INFO] Kích thước train_df: {len(train_df)}')
print(f'[INFO] Kích thước eval_df: {len(eval_df)}')
print('[INFO] Phân phối nhãn train:')
print(train_df['complaint_label'].value_counts(normalize=True))
print('[INFO] Phân phối nhãn eval:')
print(eval_df['complaint_label'].value_counts(normalize=True))

# Chuyển sang Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df[['cleaned_text', 'complaint_label']].rename(columns={'cleaned_text': 'text'}).reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df[['cleaned_text', 'complaint_label']].rename(columns={'cleaned_text': 'text'}).reset_index(drop=True))
print('[OK] Đã chuyển DataFrame sang Hugging Face Dataset')

## Bước 3: Tokenization với PhoBERT

In [ ]:
# Load tokenizer PhoBERT
model_name = 'vinai/phobert-base-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
print(f'[OK] Đã load tokenizer: {model_name}')

# Hàm tokenize cho Hugging Face Dataset
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128,
    )

# Tokenize dataset và đổi tên cột nhãn thành labels
train_dataset = train_dataset.rename_column('complaint_label', 'labels')
eval_dataset = eval_dataset.rename_column('complaint_label', 'labels')

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Loại bỏ các cột không cần thiết cho Trainer
train_dataset = train_dataset.remove_columns(['text'])
eval_dataset = eval_dataset.remove_columns(['text'])

print('[OK] Tokenization hoàn tất')
print(train_dataset[0])

## Bước 4: Định nghĩa mô hình và hàm tính metric

In [ ]:
# Load PhoBERT cho bài toán sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)
print('[OK] Đã load model PhoBERT cho classification')

# Hàm tính metric theo yêu cầu

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='macro',
        zero_division=0,
    )

    return {
        'accuracy': accuracy,
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1,
    }

print('[OK] Đã định nghĩa hàm compute_metrics')

## Bước 5: Cấu hình và huấn luyện bằng Trainer

In [ ]:
# Cấu hình arguments cho Trainer
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    report_to='none',
)

# Data collator để đảm bảo batch padding đúng chuẩn
# Dù đã padding='max_length', collator vẫn an toàn cho Trainer

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('[OK] Đã khởi tạo Trainer')
print(training_args)

In [ ]:
# Huấn luyện mô hình
print('[INFO] Bắt đầu fine-tune PhoBERT...')
trainer.train()
print('[OK] Hoàn tất huấn luyện')

## Bước 6: Đánh giá và trực quan hóa

In [ ]:
# Dự đoán trên tập eval
pred_output = trainer.predict(eval_dataset)
pred_logits = pred_output.predictions
pred_labels = np.argmax(pred_logits, axis=-1)
true_labels = pred_output.label_ids

# Classification report
print('\n' + '=' * 80)
print('CLASSIFICATION REPORT - PhoBERT')
print('=' * 80)
print(classification_report(true_labels, pred_labels, target_names=['Bình thường (0)', 'Khiếu nại (1)']))
print('=' * 80)

# Accuracy tổng thể
acc = accuracy_score(true_labels, pred_labels)
print(f'[INFO] Accuracy: {acc:.4f}')

# Confusion matrix
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Bình thường (0)', 'Khiếu nại (1)'],
    yticklabels=['Bình thường (0)', 'Khiếu nại (1)'],
    linewidths=2,
    linecolor='black',
    cbar_kws={'label': 'Số lượng mẫu'}
)
plt.title('Ma Trận Nhầm Lẫn - PhoBERT', fontsize=14, fontweight='bold')
plt.xlabel('Dự đoán', fontsize=12, fontweight='bold')
plt.ylabel('Nhãn thực tế', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/phobert_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print('[OK] Đã lưu confusion matrix tại data/processed/phobert_confusion_matrix.png')